In [ ]:
!pip install gurobipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.3/14.3 MB 83.8 MB/s eta 0:00:00


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving roi_company1.csv to roi_company1.csv


Part 1, 2 & 3

In [ ]:
import gurobipy as gp
import pandas as pd

# read data
roi = pd.read_csv("roi_company1.csv")
roi.columns = roi.columns.str.strip().str.lower()

# expected: platform, tier, lowerbound, upperbound, roi
platforms = roi["platform"].unique()

# model
m = gp.Model()

# decision vars: x[p,t] = dollars (in M$) in tier t for platform p
x = {}
for p in platforms:
    for _, r in roi[roi["platform"] == p].iterrows():
        t = int(r["tier"])
        lb, ub = float(r["lowerbound"]), r["upperbound"]
        cap = float(ub) - lb if pd.api.types.is_number(ub) else gp.GRB.INFINITY
        x[p, t] = m.addVar(lb=0, ub=cap)

# total budget <= 10
m.addConstr(gp.quicksum(v for v in x.values()) <= 10)

# print + tv <= facebook + email
m.addConstr(
    gp.quicksum(x[p, t] for (p, t) in x if p in {"print", "tv"}) <=
    gp.quicksum(x[p, t] for (p, t) in x if p in {"facebook", "email"})
)

# social >= 2 * (seo + adwords)
social = {"facebook", "linkedin", "instagram", "snapchat", "twitter"}
m.addConstr(
    gp.quicksum(x[p, t] for (p, t) in x if p in social) >=
    2 * gp.quicksum(x[p, t] for (p, t) in x if p in {"seo", "adwords"})
)


# <= 3M per platform
for platform in platforms:
    m.addConstr(gp.quicksum(x[p, t] for (p, t) in x if p == platform) <= 3)


# objective: maximize total roi * spend
m.setObjective(gp.quicksum(roi.loc[(roi["platform"]==p) & (roi["tier"]==t),"roi"].iloc[0] * x[p,t] for (p,t) in x),gp.GRB.MAXIMIZE
)

# solve
m.Params.OutputFlag = 0
m.optimize()


#Output Format
def money(x): return f"${x:.2f}M"

total_invest = 0
total_return = 0
platform_summary = {}

# gather results
for (p, t), var in x.items():
    if var.X > 0:
        r_val = roi[(roi.platform == p) & (roi.tier == t)].roi.iloc[0]
        invest = var.X
        ret = invest * r_val
        platform_summary[p] = platform_summary.get(p, 0) + ret
        total_invest += invest
        total_return += ret

print("\n================== Optimal Solution ==================")
print(f"Total Budget Used: {money(total_invest)}")
print(f"Total Expected Return: {money(total_return)}")
print(f"Overall ROI: {100 * total_return / total_invest:.2f}%\n")

print("Platform Breakdown:")
print("-------------------------------------------------------")
for p in sorted(platform_summary.keys()):
    invest = sum(var.X for (pp, tt), var in x.items() if pp == p)
    ret = platform_summary[p]
    roi_pct = 100 * ret / invest
    print(f"{p:12s} | Invested: {money(invest):>6s} | Return: {money(ret):>6s} | ROI: {roi_pct:5.2f}%")

print("=======================================================\n")

Restricted license - for non-production use only - expires 2026-11-23

================== Optimal Solution ==================
Total Budget Used: $10.00M
Total Expected Return: $0.54M
Overall ROI: 5.44%

Platform Breakdown:
-------------------------------------------------------
AdWords      | Invested: $1.00M | Return: $0.04M | ROI:  4.19%
Email        | Invested: $3.00M | Return: $0.15M | ROI:  4.93%
Instagram    | Invested: $3.00M | Return: $0.17M | ROI:  5.71%
TV           | Invested: $3.00M | Return: $0.18M | ROI:  6.08%



Part 4

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving roi_company2.csv to roi_company2.csv


In [ ]:
import numpy as np

#read in 2nd data csv
roi2 = pd.read_csv("roi_company2.csv")
roi2.columns = roi2.columns.str.strip().str.lower()

#get platforms
platforms2 = roi2["platform"].unique()

#create MIP model
m2 = gp.Model()

#decision variables
x2 = {}  #x[p,t] = amount invested in tier t for platform p
z = {}   #z[p,t] = 1 if we use tier t for platform p

for p in platforms2:
    for _, r in roi2[roi2["platform"] == p].iterrows():
        t = int(r["tier"])
        lb = float(r["lowerbound"])

        #check if UB is valid
        ub_val = r["upperbound"]
        ub_str = str(ub_val).strip().lower()

        if ub_str in ['inf', 'infinity', 'nan', ''] or pd.isna(ub_val):
            cap = 3.0
        else:
            try:
                ub = float(ub_val)
                cap = ub - lb
                if cap <= 0 or not np.isfinite(cap):
                    cap = 3.0
            except:
                cap = 3.0

        x2[p, t] = m2.addVar(lb=0, ub=cap, name=f"x_{p}_{t}")
        z[p, t] = m2.addVar(vtype=gp.GRB.BINARY, name=f"z_{p}_{t}")

#linking constraints
for p in platforms2:
    tier_list = sorted([int(r["tier"]) for _, r in roi2[roi2["platform"] == p].iterrows()])

    for t in tier_list:
        r = roi2[(roi2["platform"] == p) & (roi2["tier"] == t)].iloc[0]
        lb = float(r["lowerbound"])

        ub_val = r["upperbound"]
        ub_str = str(ub_val).strip().lower()

        if ub_str in ['inf', 'infinity', 'nan', ''] or pd.isna(ub_val):
            cap = 3.0
        else:
            try:
                ub = float(ub_val)
                cap = ub - lb
                if cap <= 0 or not np.isfinite(cap):
                    cap = 3.0
            except:
                cap = 3.0

        #x[p,t] can only be positive if z[p,t] = 1
        m2.addConstr(x2[p, t] <= cap * z[p, t])

        #if next tier is used, this tier must be full
        if t < max(tier_list):
            next_t = t + 1
            m2.addConstr(x2[p, t] >= cap * z[p, next_t])

#budget constraint
m2.addConstr(gp.quicksum(x2[p, t] for (p, t) in x2) <= 10)

#helper function for case-insensitive platform matching
def get_platform_sum(platform_names):
    return gp.quicksum(
        x2[p, t] for (p, t) in x2
        if p.lower() in [name.lower() for name in platform_names]
    )

#constraint 1: print + tv <= facebook + email
m2.addConstr(
    get_platform_sum(["print", "tv"]) <= get_platform_sum(["facebook", "email"])
)

#constraint 2: social >= 2 * (seo + adwords)
m2.addConstr(
    get_platform_sum(["facebook", "linkedin", "instagram", "snapchat", "twitter"]) >=
    2 * get_platform_sum(["seo", "adwords"])
)

#constraint 3: <= 3M per platform
for platform in platforms2:
    m2.addConstr(gp.quicksum(x2[p, t] for (p, t) in x2 if p == platform) <= 3)

#objective: maximize total return
m2.setObjective(
    gp.quicksum(
        roi2.loc[(roi2["platform"]==p) & (roi2["tier"]==t), "roi"].iloc[0] * x2[p,t]
        for (p,t) in x2
    ),
    gp.GRB.MAXIMIZE
)

#solve
m2.Params.OutputFlag = 0
m2.optimize()

#print results
print("\n" + "="*20 + " Part 4: MIP Solution (roi_company2.csv) " + "="*20)
total_invest2 = sum(var.X for var in x2.values())
total_return2 = m2.objVal

print(f"Total Budget Used: {money(total_invest2)}")
print(f"Total Expected Return: {money(total_return2)}")

if total_invest2 > 0:
    print(f"Overall ROI: {100 * total_return2 / total_invest2:.2f}%\n")
else:
    print(f"Overall ROI: N/A (no investment made)\n")

print("Platform Breakdown:")
print("-" * 60)

#collect platform summary
platform_summary2 = {}
for (p, t), var in x2.items():
    if var.X > 0.001:
        r_val = roi2[(roi2["platform"] == p) & (roi2["tier"] == t)].roi.iloc[0]
        invest = var.X
        ret = invest * r_val
        platform_summary2[p] = platform_summary2.get(p, 0) + ret

#print platform totals in sorted order
for p in sorted(platform_summary2.keys()):
    invest = sum(x2[pp, tt].X for (pp, tt) in x2 if pp == p)
    ret = platform_summary2[p]
    roi_pct = 100 * ret / invest if invest > 0 else 0
    print(f"{p:12s} | Invested: {money(invest):>6s} | Return: {money(ret):>6s} | ROI: {roi_pct:>5.2f}%")

print("=" * 60 + "\n")


==================== Part 4: MIP Solution (roi_company2.csv) ====================
Total Budget Used: $10.00M
Total Expected Return: $0.45M
Overall ROI: 4.53%

Platform Breakdown:
------------------------------------------------------------
AdWords      | Invested: $2.33M | Return: $0.11M | ROI:  4.56%
Facebook     | Invested: $3.00M | Return: $0.13M | ROI:  4.33%
LinkedIn     | Invested: $1.67M | Return: $0.07M | ROI:  4.43%
Print        | Invested: $3.00M | Return: $0.14M | ROI:  4.75%



Part 5

In [ ]:
# ===== Q5 (dynamic): Cross-evaluate using your Part 3 (LP) and Part 4 (MIP) results =====
# Uses the variables you already created:
#   roi   -> Company 1 ROI table (from Part 3)
#   roi2  -> Company 2 ROI table (from Part 4)
#   x     -> {(platform, tier): Gurobi Var} from Part 3
#   x2    -> {(platform, tier): Gurobi Var} from Part 4
#
# This block:
#   1) Builds allocation_lp / allocation_mip dynamically from x / x2
#   2) Computes each plan’s value on each ROI table (piecewise across tiers)
#   3) Prints optimal objectives, cross-eval values, losses, and “are allocations the same?”

import pandas as pd
import numpy as np

# --- Build allocations from your solved models ---
def agg_allocation_from_vars(xvars):
    alloc = {}
    for (p, t), var in xvars.items():
        v = float(var.X)
        if v > 1e-9:
            alloc[p] = alloc.get(p, 0.0) + v
    return alloc

allocation_lp  = agg_allocation_from_vars(x)   # dict {platform: spend M$} from Part 3
allocation_mip = agg_allocation_from_vars(x2)  # dict {platform: spend M$} from Part 4

# --- Normalize ROI tables (light) ---
def normalize_roi(df):
    d = df.copy()
    d.columns = d.columns.str.strip()
    d.rename(columns={
        'platform':'Platform','tier':'Tier',
        'lowerbound':'LowerBound','upperbound':'UpperBound','roi':'ROI'
    }, inplace=True)
    # ensure numeric
    for col in ['LowerBound','UpperBound','ROI','Tier']:
        if col in d.columns:
            d[col] = pd.to_numeric(d[col], errors='coerce')
    # sort tiers by spend range
    if {'Platform','LowerBound'}.issubset(d.columns):
        d = d.sort_values(['Platform','LowerBound'])
    return d

roi1_norm = normalize_roi(roi)
roi2_norm = normalize_roi(roi2)

# --- Piecewise evaluator: flow a platform’s spend through its tiers in order ---
def compute_return(roi_df, allocation):
    total_return = 0.0
    # pre-normalize platform names for case-insensitive matching
    df = roi_df.copy()
    df['Platform_norm'] = df['Platform'].astype(str).str.lower()
    for platform, invest in allocation.items():
        if invest <= 0:
            continue
        pkey = str(platform).lower()
        dfp = df[df['Platform_norm'] == pkey]
        if dfp.empty:
            continue
        remaining = float(invest)
        for _, row in dfp.iterrows():
            lb = float(row['LowerBound'])
            ub = row['UpperBound']
            r  = float(row['ROI'])
            # tier capacity = (UB - LB); if UB is NaN/±inf, treat as open-ended for cross-eval
            if pd.isna(ub) or not np.isfinite(float(ub)):
                tier_cap = remaining
            else:
                tier_cap = max(0.0, float(ub) - lb)
            take = min(remaining, tier_cap)
            if take > 0:
                total_return += take * r
                remaining -= take
            if remaining <= 1e-6:
                break
    return total_return

# --- Compute optimal objectives from your actual allocations (no hard-coding) ---
obj_lp  = compute_return(roi1_norm, allocation_lp)     # Company1 optimal objective (Part 3 plan on ROI1)
obj_mip = compute_return(roi2_norm, allocation_mip)    # Company2 optimal objective (Part 4 plan on ROI2)

# --- Cross-evaluation ---
ret_lp_on_roi2  = compute_return(roi2_norm, allocation_lp)   # LP plan on ROI2
ret_mip_on_roi1 = compute_return(roi1_norm, allocation_mip)  # MIP plan on ROI1

loss_lp_under_roi2      = obj_mip - ret_lp_on_roi2
rel_loss_lp_under_roi2  = 100 * loss_lp_under_roi2 / obj_mip if obj_mip > 0 else 0.0

loss_mip_under_roi1     = obj_lp - ret_mip_on_roi1
rel_loss_mip_under_roi1 = 100 * loss_mip_under_roi1 / obj_lp if obj_lp > 0 else 0.0

# --- Are allocations the same? ---
same_alloc = all(abs(allocation_lp.get(k,0.0) - allocation_mip.get(k,0.0)) < 1e-6
                 for k in set(allocation_lp) | set(allocation_mip))

# --- Print results (friend-style formatting) ---
print("Cross-evaluation results")
print(f"Optimal ROI1 objective: {obj_lp:.6f}")
print(f"Optimal ROI2 objective: {obj_mip:.6f}\n")

print(f"ROI1 optimal allocation used on ROI2 data: {ret_lp_on_roi2:.6f}")
print(f"Loss: {loss_lp_under_roi2:.6f} ({rel_loss_lp_under_roi2:.2f}% lower)\n")

print(f"ROI2 optimal allocation used on ROI1 data: {ret_mip_on_roi1:.6f}")
print(f"Loss: {loss_mip_under_roi1:.6f} ({rel_loss_mip_under_roi1:.2f}% lower)\n")

print(f"Are the allocations the same? {'Yes' if same_alloc else 'No'}")

Cross-evaluation results
Optimal ROI1 objective: 0.543640
Optimal ROI2 objective: 0.452827

ROI1 optimal allocation used on ROI2 data: 0.277720
Loss: 0.175107 (38.67% lower)

ROI2 optimal allocation used on ROI1 data: 0.274903
Loss: 0.268737 (49.43% lower)

Are the allocations the same? No


Part 6

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving min_amount.csv to min_amount.csv


In [ ]:
# ===== Q6: MIP with per-channel minimum spend (uses roi2 + min_amount.csv) =====
import gurobipy as gp
from gurobipy import GRB
import pandas as pd, numpy as np

# roi2 is assumed to be in memory (Company 2 ROI table)
# If not, uncomment: roi2 = pd.read_csv("roi_company2.csv")
mins = pd.read_csv("min_amount.csv")  # columns: Platform, MinInvestment

# --- Normalize inputs ---
roi2 = roi2.copy()
roi2.columns = roi2.columns.str.strip()
mins = mins.copy()
mins.columns = mins.columns.str.strip()

# Ensure expected column names
roi2.rename(columns={
    'platform':'Platform','tier':'Tier','lowerbound':'LowerBound',
    'upperbound':'UpperBound','roi':'ROI'
}, inplace=True)
mins.rename(columns={'mininvestment':'MinInvestment'}, inplace=True)

# Replace +/-inf in UpperBound with 3.0 and treat missing UB as 3.0 (platform cap)
CAP, BUDGET = 3.0, 10.0
roi2['UpperBound'] = roi2['UpperBound'].replace([np.inf, -np.inf], CAP).fillna(CAP)

# Sort tiers per platform by LowerBound
roi2 = roi2.sort_values(['Platform','LowerBound'])

# Convenience: minimum spend per platform
MIN = {r.Platform: float(r["MinInvestment"]) for _, r in mins.iterrows()}

platforms  = roi2.Platform.unique().tolist()
tiers_by_p = {p: roi2[roi2.Platform==p].Tier.tolist() for p in platforms}

# Dicts for quick access
L = {(r.Platform, r.Tier): float(r.LowerBound)  for _, r in roi2.iterrows()}
U = {(r.Platform, r.Tier): float(r.UpperBound)  for _, r in roi2.iterrows()}
R = {(r.Platform, r.Tier): float(r.ROI)         for _, r in roi2.iterrows()}

# Case-insensitive name access (so constraints like "Print" work even if cases differ)
plat_key = {p.lower(): p for p in platforms}
def pk(name): return plat_key[name.lower()]  # map canonical name -> actual key

# --- NEW: precompute per-tier width and cumulative base return (correct piecewise ROI) ---
# width[(p,t)] = usable span of tier t (capped at 3M)
# base[(p,t)]  = sum over earlier tiers j of (ROI[p,j] * width[(p,j)])
WIDTH = {}
BASE  = {}
for p in platforms:
    # sort this platform's tiers by lower bound (already sorted globally, but safe)
    tlist = sorted(tiers_by_p[p], key=lambda tt: L[(p,tt)])
    cum = 0.0
    for t in tlist:
        w = max(0.0, min(U[(p,t)], CAP) - L[(p,t)])
        WIDTH[(p,t)] = w
        BASE[(p,t)]  = cum
        cum += R[(p,t)] * w

# --- Model ---
m = gp.Model("Q6_MinimumSpend")
m.Params.OutputFlag = 0

# Decision variables
# x[p]  = total spend (M$) for platform p
# z[p,t]= 1 if tier t is selected for platform p (at most one tier per platform)
# s[p,t]= fraction in (U[p,t]-L[p,t]) for platform p's chosen tier t (0..1)
# y[p]  = 1 if platform p is invested in (to trigger min spend and max cap)
x = {p: m.addVar(lb=0.0, ub=CAP, name=f"x[{p}]") for p in platforms}
z = {(p,t): m.addVar(vtype=GRB.BINARY, name=f"z[{p},{t}]")
     for p in platforms for t in tiers_by_p[p]}
s = {(p,t): m.addVar(lb=0.0, ub=1.0, name=f"s[{p},{t}]")
     for p in platforms for t in tiers_by_p[p]}
y = {p: m.addVar(vtype=GRB.BINARY, name=f"y[{p}]") for p in platforms}

m.update()

# (a) Tier selection and spend composition per platform
for p in platforms:
    # At most one tier active
    m.addConstr(gp.quicksum(z[p,t] for t in tiers_by_p[p]) <= 1, name=f"one_tier[{p}]")
    # Total spend equals LB + fraction*(UB-LB) for the active tier (or 0 if none)
    m.addConstr(
        x[p] == gp.quicksum(L[p,t]*z[p,t] + (U[p,t]-L[p,t])*s[p,t] for t in tiers_by_p[p]),
        name=f"compose[{p}]"
    )
    # Fraction only if the tier is selected
    for t in tiers_by_p[p]:
        m.addConstr(s[p,t] <= z[p,t], name=f"link_s_z[{p},{t}]")

# (b) Budget
m.addConstr(gp.quicksum(x[p] for p in platforms) <= BUDGET, name="budget")

# (c) Boss's constraints
m.addConstr(x[pk("Print")] + x[pk("TV")] <= x[pk("Facebook")] + x[pk("Email")],
            name="rule_print_tv_vs_fb_email")

social = x[pk("Facebook")] + x[pk("LinkedIn")] + x[pk("Instagram")] + x[pk("Snapchat")] + x[pk("Twitter")]
search = x[pk("SEO")] + x[pk("AdWords")]
m.addConstr(social >= 2.0 * search, name="rule_social_vs_search")

# (d) Minimum-spend if invested (and platform cap via x upper bound)
for p in platforms:
    m.addConstr(x[p] >= MIN.get(p, 0.0) * y[p], name=f"min_if_on[{p}]")
    m.addConstr(x[p] <= CAP * y[p],           name=f"cap_if_on[{p}]")

# === CHANGED: Objective (correct piecewise valuation)
# old: sum R[p,t] * ( L[p,t]*z + (U-L)*s )
# new: sum BASE[p,t] * z[p,t]  +  R[p,t] * ( (U-L) * s[p,t] )
m.setObjective(
    gp.quicksum(BASE[(p,t)] * z[p,t] + R[(p,t)] * ((U[(p,t)] - L[(p,t)]) * s[p,t])
                for p in platforms for t in tiers_by_p[p]),
    GRB.MAXIMIZE
)

m.optimize()

# --- Output (unchanged) ---
if m.status == GRB.OPTIMAL:
    total_spend  = sum(x[p].X for p in platforms)
    total_return = m.objVal
    roi_percent  = 100.0 * total_return / total_spend if total_spend > 0 else 0.0

    print("\n=== Q6: Optimal Allocation with Minimum Spend ===")
    print(f"Total Budget Available: ${BUDGET:6.3f}M")
    print(f"Total Spend Used:       ${total_spend:6.3f}M")
    print(f"Total Return Achieved:  ${total_return:6.3f}M")
    print(f"Overall ROI:            {roi_percent:5.2f}%\n")

    print(f"{'Platform':<12}{'Invest?':<10}{'Spend($M)':<12}{'Min Req($M)':<14}{'Active Tier':<10}")
    print("-"*60)
    for p in platforms:
        spend = x[p].X
        invest_flag = "Yes" if y[p].X > 0.5 else "No"
        chosen_tier = [t for t in tiers_by_p[p] if z[p,t].X > 0.5]
        tier_display = chosen_tier[0] if chosen_tier else "-"
        print(f"{p:<12}{invest_flag:<10}{spend:<12.3f}{MIN.get(p,0.0):<14.3f}{tier_display:<10}")

    print("-"*60)
    print("Constraint check:")
    print(f"  Total Spend ≤ {BUDGET:5.3f}:       {total_spend:5.3f} ≤ {BUDGET:5.3f}")
    print(f"  Print+TV ≤ Facebook+Email:  {(x[pk('Print')].X + x[pk('TV')].X):5.3f} ≤ {(x[pk('Facebook')].X + x[pk('Email')].X):5.3f}")
    print(f"  Social ≥ 2×Search:          {social.getValue():5.3f} ≥ {(2*search.getValue()):5.3f}")
else:
    print("Model did not reach optimal solution.")


=== Q6: Optimal Allocation with Minimum Spend ===
Total Budget Available: $10.000M
Total Spend Used:       $10.000M
Total Return Achieved:  $ 0.453M
Overall ROI:             4.53%

Platform    Invest?   Spend($M)   Min Req($M)   Active Tier
------------------------------------------------------------
AdWords     Yes       2.333       0.800         2         
Email       No        0.000       0.500         1         
Facebook    Yes       3.000       0.400         1         
Instagram   No        0.000       0.700         1         
LinkedIn    Yes       1.667       0.200         1         
Print       Yes       3.000       0.300         2         
SEO         No        0.000       0.600         1         
Snapchat    No        0.000       0.800         1         
TV          No        0.000       0.300         1         
Twitter     No        0.000       0.300         1         
------------------------------------------------------------
Constraint check:
  Total Spend ≤ 10.000:     

Part 7

In [ ]:
from google.colab import files
print("Please upload roi_monthly.csv")
uploaded = files.upload()

Please upload roi_monthly.csv


Saving roi_monthly.csv to roi_monthly.csv


In [ ]:
# Q7 - Imports and data load
# Try gurobipy; install if missing
try:
    import gurobipy as gp
    from gurobipy import GRB
except Exception as e:
    print("Installing gurobipy...")
    !pip -q install gurobipy
    import gurobipy as gp
    from gurobipy import GRB

import pandas as pd
import numpy as np
import math

# Read and normalize columns (accept both LowerBoundM/UpperBoundM or LowerBound/UpperBound)
roi_monthly = pd.read_csv("roi_monthly.csv")
roi_monthly.columns = roi_monthly.columns.str.strip()

# Unify expected column names to lower snake
rename_map = {
    "Month": "month", "Platform": "platform", "Tier": "tier",
    "LowerBoundM": "lowerboundm", "UpperBoundM": "upperboundm",
    "LowerBound": "lowerboundm",  "UpperBound": "upperboundm",  # fallback, just in case
    "ROI": "roi"
}
roi_monthly = roi_monthly.rename(columns={k: v for k, v in rename_map.items() if k in roi_monthly.columns})
missing = {"month","platform","tier","lowerboundm","upperboundm","roi"} - set(roi_monthly.columns.str.lower())
assert not missing, f"roi_monthly.csv missing columns: {missing}"

# Standardize platform names
roi_monthly["platform"] = roi_monthly["platform"].astype(str).str.strip()

# Constants (in $Millions)
BASE_BUDGET   = 10.0
REINVEST_RATE = 0.5
PLATFORM_CAP  = 3.0

SOCIAL = {"Facebook","LinkedIn","Instagram","Snapchat","Twitter"}
SEARCH = {"SEO","AdWords"}

# Month order
MONTH_ORDER = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
months_available = [m for m in MONTH_ORDER if m in set(roi_monthly["month"])]
assert months_available, "No valid months found in roi_monthly.csv"

def build_segments(df_m: pd.DataFrame) -> pd.DataFrame:
    """
    Convert cumulative [lowerboundm, upperboundm] to segment capacities per (platform, tier).
    If upperbound is 'inf' or np.inf, give a large cap; platform cap (3.0) will bind anyway.
    """
    segs = df_m.copy()
    # Normalize 'inf' text to np.inf
    segs["upperboundm"] = segs["upperboundm"].replace({"inf": np.inf, "Inf": np.inf, "INF": np.inf})
    # Compute segment width
    def seg_cap(lb, ub):
        if pd.isna(ub):  # be defensive
            return 0.0
        if np.isfinite(ub):
            return max(0.0, float(ub) - float(lb))
        else:
            return 10.0  # generous cap; platform cap will limit to 3.0
    segs["segcap"] = [seg_cap(lb, ub) for lb, ub in zip(segs["lowerboundm"], segs["upperboundm"])]
    segs = segs[["platform","tier","roi","segcap"]].reset_index(drop=True)
    return segs

Installing gurobipy...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.3/14.3 MB 65.2 MB/s eta 0:00:00


In [ ]:
# Q7 - Monthly LP with 50% reinvestment (print-only)

def solve_month_lp(segs: pd.DataFrame, budget: float):
    """
    Gurobi LP:
      max sum(roi_i * x_i)
      s.t.
         sum_i x_i = budget
         per-platform sum <= 3.0
         0 <= x_i <= segcap_i
         (Print + TV) <= (Facebook + Email)
         (Facebook+LinkedIn+Instagram+Snapchat+Twitter) >= 2*(SEO+AdWords)
    Returns: (revenue, platform_allocation_df)
    """
    platforms = segs["platform"].tolist()
    rois      = segs["roi"].to_numpy(dtype=float)
    caps      = segs["segcap"].to_numpy(dtype=float)
    n         = len(segs)

    # Build model
    m = gp.Model("q7_monthly_lp")
    m.Params.OutputFlag = 0  # silent

    x = m.addVars(n, lb=0.0, name="x")  # continuous

    # Objective
    m.setObjective(gp.quicksum(rois[i] * x[i] for i in range(n)), GRB.MAXIMIZE)

    # Budget equality
    m.addConstr(gp.quicksum(x[i] for i in range(n)) == budget, name="budget")

    # Segment bounds: x_i <= segcap_i
    for i in range(n):
        m.addConstr(x[i] <= float(caps[i]), name=f"cap_{i}")

    # Per-platform cap
    for p in segs["platform"].unique():
        idxs = [i for i in range(n) if platforms[i] == p]
        m.addConstr(gp.quicksum(x[i] for i in idxs) <= PLATFORM_CAP, name=f"platcap_{p}")

    # Boss #1: Print + TV <= Facebook + Email
    idx_print_tv = [i for i in range(n) if platforms[i] in {"Print","TV"}]
    idx_fb_email = [i for i in range(n) if platforms[i] in {"Facebook","Email"}]
    m.addConstr(gp.quicksum(x[i] for i in idx_print_tv)
                <= gp.quicksum(x[i] for i in idx_fb_email), name="boss1")

    # Boss #2: Social >= 2*(Search)
    idx_social = [i for i in range(n) if platforms[i] in SOCIAL]
    idx_search = [i for i in range(n) if platforms[i] in SEARCH]
    m.addConstr(gp.quicksum(x[i] for i in idx_social)
                >= 2.0 * gp.quicksum(x[i] for i in idx_search), name="boss2")

    m.optimize()

    if m.Status != GRB.OPTIMAL:
        raise RuntimeError(f"LP not optimal. Status: {m.Status}")

    x_val = np.array([x[i].X for i in range(n)], dtype=float)
    revenue = float(np.dot(rois, x_val))

    alloc = segs.copy()
    alloc["investment"] = x_val
    plat_alloc = alloc.groupby("platform", as_index=False)["investment"].sum()

    return revenue, plat_alloc.sort_values("platform").reset_index(drop=True)

# Roll-forward loop (print only)
current_budget = BASE_BUDGET
summary_rows = []
print("===== Q7: Monthly Optimization with 50% Reinvestment (print only) =====\n")

for mth in months_available:
    df_m = roi_monthly.loc[roi_monthly["month"] == mth].copy()
    segs = build_segments(df_m)

    revenue, plat_alloc = solve_month_lp(segs, current_budget)
    next_budget = BASE_BUDGET + REINVEST_RATE * revenue

    # Pretty print for this month
    print(f"[{mth}] Budget = ${current_budget:,.3f}M")
    print(f"[{mth}] Optimal Revenue = ${revenue:,.3f}M")
    print(f"[{mth}] Next Month Budget = 10.0 + 0.5 * {revenue:,.3f} = ${next_budget:,.3f}M")
    print("\nPer-Platform Allocation ($M):")
    # nice, compact table
    print(plat_alloc.rename(columns={"platform":"Platform","investment":"Investment($M)"}).to_string(index=False))
    print("-" * 70)

    summary_rows.append({
        "Month": mth,
        "Budget($M)": round(current_budget, 6),
        "Revenue($M)": round(revenue, 6),
        "NextBudget($M)": round(next_budget, 6),
    })
    current_budget = next_budget  # roll forward

# Final summary
summary_df = pd.DataFrame(summary_rows, columns=["Month","Budget($M)","Revenue($M)","NextBudget($M)"])
print("\n===== Q7 Summary (12 months) =====")
print(summary_df.to_string(index=False))

===== Q7: Monthly Optimization with 50% Reinvestment (print only) =====

Restricted license - for non-production use only - expires 2026-11-23
[Jan] Budget = $10.000M
[Jan] Optimal Revenue = $0.560M
[Jan] Next Month Budget = 10.0 + 0.5 * 0.560 = $10.280M

Per-Platform Allocation ($M):
 Platform  Investment($M)
  AdWords             0.0
    Email             0.0
 Facebook             3.0
Instagram             0.0
 LinkedIn             3.0
    Print             3.0
      SEO             0.0
 Snapchat             0.0
       TV             0.0
  Twitter             1.0
----------------------------------------------------------------------
[Feb] Budget = $10.280M
[Feb] Optimal Revenue = $0.454M
[Feb] Next Month Budget = 10.0 + 0.5 * 0.454 = $10.227M

Per-Platform Allocation ($M):
 Platform  Investment($M)
  AdWords        2.660083
    Email        0.000000
 Facebook        2.300000
Instagram        0.000000
 LinkedIn        0.500000
    Print        2.300000
      SEO        0.000000
 Snapc

Part 8

In [ ]:
# ============================
# Q8 — Stability Check (print-only)
# Verifies whether month-to-month platform allocations satisfy:
#     | spend_{p,t} - spend_{p,t-1} | <= 1.0 ($M)
# Reuses: roi_monthly, months_available, build_segments, solve_month_lp,
#         BASE_BUDGET, REINVEST_RATE
# ============================

import numpy as np
import pandas as pd

# 1) Re-run the monthly optimization and STORE allocations
alloc_by_month = {}   # { 'Jan': DataFrame(platform, investment), ... }
budget_by_month = {}
revenue_by_month = {}

current_budget = BASE_BUDGET

for mth in months_available:
    df_m = roi_monthly.loc[roi_monthly["month"] == mth].copy()
    segs = build_segments(df_m)
    revenue, plat_alloc = solve_month_lp(segs, current_budget)

    # Ensure all platforms appear in each month (fill missing with 0)
    all_platforms = sorted(roi_monthly["platform"].unique())
    plat_alloc = (
        plat_alloc.set_index("platform")
        .reindex(all_platforms, fill_value=0.0)
        .reset_index()
    )

    alloc_by_month[mth] = plat_alloc
    budget_by_month[mth] = current_budget
    revenue_by_month[mth] = revenue

    current_budget = BASE_BUDGET + REINVEST_RATE * revenue

# 2) Check consecutive months for stability
THRESHOLD = 1.0  # $M
violations = []  # list of tuples: (prev_month, curr_month, platform, prev, curr, abs_diff)

print("===== Q8: Stability Check (|Δ| <= $1.0M) =====\n")

for i in range(1, len(months_available)):
    prev_m = months_available[i - 1]
    curr_m = months_available[i]

    df_prev = alloc_by_month[prev_m].set_index("platform")
    df_curr = alloc_by_month[curr_m].set_index("platform")

    print(f"--- {prev_m} → {curr_m} ---")
    for plat in df_curr.index:
        prev_val = float(df_prev.loc[plat, "investment"])
        curr_val = float(df_curr.loc[plat, "investment"])
        diff = curr_val - prev_val
        adiff = abs(diff)

        flag = "OK"
        if adiff > THRESHOLD + 1e-9:
            flag = "VIOLATION"
            violations.append((prev_m, curr_m, plat, prev_val, curr_val, adiff))

        print(f"{plat:10s}: {prev_val:5.2f} → {curr_val:5.2f}   Δ={diff:5.2f}   |Δ|={adiff:5.2f}   {flag}")
    print()

# 3) Summary
if not violations:
    print("All month-to-month platform changes are within $1.0M (stable under this rule).")
else:
    print("⚠ Stability violations found (|Δ| > $1.0M):")
    for pm, cm, plat, pv, cv, ad in violations:
        print(f"  {pm} → {cm} | {plat}: {pv:.2f} → {cv:.2f}  |Δ|={ad:.2f} > 1.0M")

print("\nNote:")
print("- This cell only CHECKS stability using Q7 allocations; it does not modify the model.")
print("- To ENFORCE stability in the optimization, you would add constraints:")
print("    |spend_{p,t} - spend_{p,t-1}| <= 1.0 for all platforms p and months t.")
print("  (Implement with standard absolute-value linearization using two nonnegative slacks.)")

===== Q8: Stability Check (|Δ| <= $1.0M) =====

--- Jan → Feb ---
AdWords   :  0.00 →  2.66   Δ= 2.66   |Δ|= 2.66   VIOLATION
Email     :  0.00 →  0.00   Δ= 0.00   |Δ|= 0.00   OK
Facebook  :  3.00 →  2.30   Δ=-0.70   |Δ|= 0.70   OK
Instagram :  0.00 →  0.00   Δ= 0.00   |Δ|= 0.00   OK
LinkedIn  :  3.00 →  0.50   Δ=-2.50   |Δ|= 2.50   VIOLATION
Print     :  3.00 →  2.30   Δ=-0.70   |Δ|= 0.70   OK
SEO       :  0.00 →  0.00   Δ= 0.00   |Δ|= 0.00   OK
Snapchat  :  0.00 →  2.52   Δ= 2.52   |Δ|= 2.52   VIOLATION
TV        :  0.00 →  0.00   Δ= 0.00   |Δ|= 0.00   OK
Twitter   :  1.00 →  0.00   Δ=-1.00   |Δ|= 1.00   OK

--- Feb → Mar ---
AdWords   :  2.66 →  2.57   Δ=-0.09   |Δ|= 0.09   OK
Email     :  0.00 →  0.00   Δ= 0.00   |Δ|= 0.00   OK
Facebook  :  2.30 →  2.53   Δ= 0.23   |Δ|= 0.23   OK
Instagram :  0.00 →  0.00   Δ= 0.00   |Δ|= 0.00   OK
LinkedIn  :  0.50 →  2.60   Δ= 2.10   |Δ|= 2.10   VIOLATION
Print     :  2.30 →  2.53   Δ= 0.23   |Δ|= 0.23   OK
SEO       :  0.00 →  0.00   Δ= 0.00   |

In [ ]:
# Q8 — Enforced Stability Solver (print-only)

THRESHOLD = 1.0  # $M stability limit per platform per month

def solve_month_lp_with_stability(segs: pd.DataFrame, budget: float,
                                  prev_platform_alloc: pd.DataFrame | None,
                                  threshold: float = THRESHOLD):
    """
    Gurobi LP with stability:
      max sum(roi_i * x_i)
      s.t.
         sum_i x_i = budget
         per-platform sum <= 3.0
         0 <= x_i <= segcap_i
         Boss #1: (Print + TV) <= (Facebook + Email)
         Boss #2: (Facebook+LinkedIn+Instagram+Snapchat+Twitter) >= 2*(SEO + AdWords)
         Stability (if prev_platform_alloc is given):
             | sum_{i in p} x_i  -  prev_alloc[p] | <= threshold
             implemented via s_plus_p, s_minus_p >= 0:
                 (sum_{i in p} x_i) - prev_alloc[p] = s_plus_p - s_minus_p
                 s_plus_p + s_minus_p <= threshold
    Returns: (revenue, platform_allocation_df)
    """
    import gurobipy as gp
    from gurobipy import GRB
    import numpy as np

    platforms = segs["platform"].tolist()
    rois      = segs["roi"].to_numpy(dtype=float)
    caps      = segs["segcap"].to_numpy(dtype=float)
    n         = len(segs)

    m = gp.Model("q8_monthly_lp_stability")
    m.Params.OutputFlag = 0  # silent

    # Decision variables: segment investments
    x = m.addVars(n, lb=0.0, name="x")  # continuous

    # Objective
    m.setObjective(gp.quicksum(rois[i] * x[i] for i in range(n)), GRB.MAXIMIZE)

    # Budget equality
    m.addConstr(gp.quicksum(x[i] for i in range(n)) == budget, name="budget")

    # Segment caps
    for i in range(n):
        m.addConstr(x[i] <= float(caps[i]), name=f"cap_{i}")

    # Per-platform cap: <= 3.0
    unique_plats = list(segs["platform"].unique())
    plat_sum = {}  # store expressions per platform for re-use
    for p in unique_plats:
        idxs = [i for i in range(n) if platforms[i] == p]
        expr = gp.quicksum(x[i] for i in idxs)
        plat_sum[p] = expr
        m.addConstr(expr <= 3.0, name=f"platcap_{p}")

    # Boss #1: Print + TV <= Facebook + Email
    expr_print_tv = gp.quicksum(plat_sum.get(p, 0) for p in ["Print", "TV"])
    expr_fb_email = gp.quicksum(plat_sum.get(p, 0) for p in ["Facebook", "Email"])
    m.addConstr(expr_print_tv <= expr_fb_email, name="boss1")

    # Boss #2: Social >= 2 * Search
    SOCIAL = {"Facebook","LinkedIn","Instagram","Snapchat","Twitter"}
    SEARCH = {"SEO","AdWords"}
    expr_social = gp.quicksum(plat_sum.get(p, 0) for p in SOCIAL if p in plat_sum)
    expr_search = gp.quicksum(plat_sum.get(p, 0) for p in SEARCH if p in plat_sum)
    m.addConstr(expr_social >= 2.0 * expr_search, name="boss2")

    # Stability constraints (from month 2 onward)
    if prev_platform_alloc is not None:
        prev_map = prev_platform_alloc.set_index("platform")["investment"].to_dict()
        for p in unique_plats:
            prev_val = float(prev_map.get(p, 0.0))
            s_plus  = m.addVar(lb=0.0, name=f"s_plus[{p}]")
            s_minus = m.addVar(lb=0.0, name=f"s_minus[{p}]")
            # sum_p - prev = s_plus - s_minus
            m.addConstr(plat_sum[p] - prev_val == s_plus - s_minus, name=f"stab_balance[{p}]")
            # s_plus + s_minus <= threshold
            m.addConstr(s_plus + s_minus <= threshold, name=f"stab_limit[{p}]")

    m.optimize()
    if m.Status != GRB.OPTIMAL:
        raise RuntimeError(f"LP not optimal. Status: {m.Status}")

    x_val = np.array([x[i].X for i in range(n)], dtype=float)
    revenue = float(np.dot(rois, x_val))

    alloc = segs.copy()
    alloc["investment"] = x_val
    plat_alloc = alloc.groupby("platform", as_index=False)["investment"].sum()

    return revenue, plat_alloc.sort_values("platform").reset_index(drop=True)

In [ ]:
# Re-solve the 12-month sequence with stability constraints enforced
current_budget = BASE_BUDGET
alloc_by_month_stab = {}
budgets_stab, revenues_stab, next_budgets_stab = [], [], []

prev_alloc = None
print("===== Q8: Monthly Optimization WITH Stability (|Δ| <= $1.0M) =====\n")

# we’ll use the union of platforms across the file to keep tables aligned
all_platforms = sorted(roi_monthly["platform"].unique())

for mth in months_available:
    df_m = roi_monthly.loc[roi_monthly["month"] == mth].copy()
    segs = build_segments(df_m)

    rev, plat_alloc = solve_month_lp_with_stability(segs, current_budget, prev_alloc, threshold=THRESHOLD)

    # normalize platform rows
    plat_alloc = plat_alloc.set_index("platform").reindex(all_platforms, fill_value=0.0).reset_index()

    # print month result
    next_budget = BASE_BUDGET + REINVEST_RATE * rev
    print(f"[{mth}] Budget = ${current_budget:,.3f}M")
    print(f"[{mth}] Optimal Revenue (stability-enforced) = ${rev:,.3f}M")
    print(f"[{mth}] Next Month Budget = 10.0 + 0.5 * {rev:,.3f} = ${next_budget:,.3f}M")
    print("\nPer-Platform Allocation ($M) — Stability ON:")
    print(plat_alloc.rename(columns={"platform":"Platform","investment":"Investment($M)"}).to_string(index=False))
    print("-" * 70)

    # store & roll
    alloc_by_month_stab[mth] = plat_alloc
    budgets_stab.append(current_budget)
    revenues_stab.append(rev)
    next_budgets_stab.append(next_budget)
    current_budget = next_budget
    prev_alloc = plat_alloc  # for next month's stability constraints

# Quick verification: check there are no stability violations now
print("\n===== Verification: Stability Check (should be no violations) =====\n")
violations = []
for i in range(1, len(months_available)):
    pm, cm = months_available[i-1], months_available[i]
    df_prev = alloc_by_month_stab[pm].set_index("platform")
    df_curr = alloc_by_month_stab[cm].set_index("platform")
    for plat in all_platforms:
        diff = float(df_curr.loc[plat, "investment"] - df_prev.loc[plat, "investment"])
        if abs(diff) > THRESHOLD + 1e-6:
            violations.append((pm, cm, plat, float(df_prev.loc[plat, "investment"]),
                               float(df_curr.loc[plat, "investment"]), abs(diff)))

if not violations:
    print("All month-to-month platform changes are within $1.0M. (OK)")
else:
    print("⚠ Violations found (unexpected):")
    for v in violations:
        pm, cm, plat, pv, cv, ad = v
        print(f"  {pm} → {cm} | {plat}: {pv:.2f} → {cv:.2f}  |Δ|={ad:.2f} > 1.0M")

# Optional: summary line
print("\nDone. You now have a stability-enforced allocation for every month.")

===== Q8: Monthly Optimization WITH Stability (|Δ| <= $1.0M) =====

[Jan] Budget = $10.000M
[Jan] Optimal Revenue (stability-enforced) = $0.560M
[Jan] Next Month Budget = 10.0 + 0.5 * 0.560 = $10.280M

Per-Platform Allocation ($M) — Stability ON:
 Platform  Investment($M)
  AdWords             0.0
    Email             0.0
 Facebook             3.0
Instagram             0.0
 LinkedIn             3.0
    Print             3.0
      SEO             0.0
 Snapchat             0.0
       TV             0.0
  Twitter             1.0
----------------------------------------------------------------------
[Feb] Budget = $10.280M
[Feb] Optimal Revenue (stability-enforced) = $0.420M
[Feb] Next Month Budget = 10.0 + 0.5 * 0.420 = $10.210M

Per-Platform Allocation ($M) — Stability ON:
 Platform  Investment($M)
  AdWords         1.00000
    Email         0.18025
 Facebook         3.00000
Instagram         0.00000
 LinkedIn         2.10000
    Print         3.00000
      SEO         0.00000
 Snapchat